# Subtask 2 — Attention + Decoding Strategies

## Goal

This experiment extends the basic LSTM encoder-decoder from Subtask 1 by adding an **attention mechanism**.

The dataset and translation task remain exactly the same as Subtask 1:

**English → French translation**

The main objective is to determine whether attention improves translation quality by allowing the decoder to access all encoder states instead of relying only on the encoder's final hidden and cell states.

### Subtask 1

The basic model uses:

```text
English sentence
       ↓
Encoder LSTM
       ↓
Final hidden + cell states
       ↓
Decoder LSTM
       ↓
French translation

1) We will use Bahdanau (additive) attention because it directly addresses the failure we observed in subtask 1
2) The decoder shouldn't have to depend only on the encoder's final state

# Subtask 2

```text
English sentence
       ↓
Encoder LSTM
       ↓
All encoder hidden states
       ↓
Attention mechanism
       ↑
Decoder state
       ↓
Context vector
       ↓
Decoder
       ↓
French translation

In [2]:
# Load the same English-French datasets used in Subtask 1

import pandas as pd
import tensorflow as tf

train_df = pd.read_csv("/content/en_fr_train.csv")
val_df = pd.read_csv("/content/en_fr_val.csv")
test_df = pd.read_csv("/content/en_fr_test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (232825, 2)
Validation: (890, 2)
Test: (8597, 2)


## Dataset and Preprocessing

The same English-French parallel corpus from Subtask 1 is used so that the effect of attention can be evaluated fairly.

| Split | Samples | Columns |
|---|---:|---|
| Train | 232,825 | `en`, `fr` |
| Validation | 890 | `en`, `fr` |
| Test | 8,597 | `en`, `fr` |

The preprocessing pipeline will also remain unchanged:

- Lowercase the text
- Tokenize words and punctuation
- Add `<SOS>` and `<EOS>` tokens
- Use `<PAD>` for sequence padding
- Use `<UNK>` for unseen/rare words
- Build vocabularies using the training set
- Convert sentences into integer token IDs

Keeping the dataset and preprocessing fixed makes the comparison between the **no-attention** and **attention** models more meaningful.

In [3]:
# Create the same tokenizer and vocabularies used in Subtask 1

import re
from collections import Counter

def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[^\w\s]", text)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
MIN_FREQ = 2

eng_counter = Counter()
fra_counter = Counter()

for text in train_df["en"]:
    eng_counter.update(tokenize(text))

for text in train_df["fr"]:
    fra_counter.update(tokenize(text))

eng_vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
fra_vocab = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

for token, count in eng_counter.items():
    if count >= MIN_FREQ:
        eng_vocab[token] = len(eng_vocab)

for token, count in fra_counter.items():
    if count >= MIN_FREQ:
        fra_vocab[token] = len(fra_vocab)

print("English vocabulary:", len(eng_vocab))
print("French vocabulary:", len(fra_vocab))
print("Special tokens:", {token: eng_vocab[token] for token in SPECIAL_TOKENS})

English vocabulary: 35309
French vocabulary: 41836
Special tokens: {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}


In [4]:
# Encode sentences into token IDs

def encode_sentence(text, vocab):
    tokens = tokenize(text)
    ids = [vocab["<SOS>"]]

    for token in tokens:
        ids.append(vocab.get(token, vocab["<UNK>"]))

    ids.append(vocab["<EOS>"])
    return ids


def encode_dataframe(df, eng_vocab, fra_vocab):
    english_sequences = []
    french_sequences = []

    for _, row in df.iterrows():
        english_sequences.append(encode_sentence(row["en"], eng_vocab))
        french_sequences.append(encode_sentence(row["fr"], fra_vocab))

    return english_sequences, french_sequences


train_en, train_fr = encode_dataframe(train_df, eng_vocab, fra_vocab)
val_en, val_fr = encode_dataframe(val_df, eng_vocab, fra_vocab)
test_en, test_fr = encode_dataframe(test_df, eng_vocab, fra_vocab)

print("Train:", len(train_en), len(train_fr))
print("Validation:", len(val_en), len(val_fr))
print("Test:", len(test_en), len(test_fr))

Train: 232825 232825
Validation: 890 890
Test: 8597 8597


## Building Attention Model (Bahdanau(additive) attention)

- For each decoder step we do:-
- ![image.png](attachment:image.png)

In [15]:
# Implement Bahdanau additive attention

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()

        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        # query: decoder hidden state -> (batch, units)
        # values: all encoder outputs -> (batch, source_length, units)

        query = tf.expand_dims(query, 1)

        score = self.V(
            tf.nn.tanh(
                self.W1(values) + self.W2(query)
            )
        )

        attention_weights = tf.nn.softmax(score, axis=1)

        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [16]:
# Build the attention Seq2Seq model with graph-compatible Bahdanau attention

EMBEDDING_DIM = 128
UNITS = 256

class AttentionSeq2Seq(tf.keras.Model):
    def __init__(self, eng_vocab_size, fra_vocab_size, embedding_dim, units):
        super().__init__()

        self.eng_embedding = tf.keras.layers.Embedding(
            eng_vocab_size, embedding_dim, mask_zero=True
        )

        self.encoder_lstm = tf.keras.layers.LSTM(
            units, return_sequences=True, return_state=True
        )

        self.fra_embedding = tf.keras.layers.Embedding(
            fra_vocab_size, embedding_dim, mask_zero=True
        )

        self.decoder_lstm = tf.keras.layers.LSTM(
            units, return_sequences=True, return_state=True
        )

        self.attention = BahdanauAttention(units)

        self.decoder_dense = tf.keras.layers.Dense(
            fra_vocab_size, activation="softmax"
        )

    def call(self, encoder_input, decoder_input):
        # Encode the English sentence
        encoder_embedded = self.eng_embedding(encoder_input)

        encoder_outputs, encoder_h, encoder_c = self.encoder_lstm(
            encoder_embedded
        )

        # Decode using teacher forcing
        decoder_embedded = self.fra_embedding(decoder_input)

        decoder_outputs, _, _ = self.decoder_lstm(
            decoder_embedded,
            initial_state=[encoder_h, encoder_c]
        )

        # Compute attention for every decoder timestep
        target_length = tf.shape(decoder_outputs)[1]

        context_array = tf.TensorArray(
            dtype=decoder_outputs.dtype,
            size=target_length
        )

        i = tf.constant(0)

        def condition(i, context_array):
            return i < target_length

        def body(i, context_array):
            context, _ = self.attention(
                decoder_outputs[:, i, :],
                encoder_outputs
            )

            context_array = context_array.write(i, context)
            return i + 1, context_array

        _, context_array = tf.while_loop(
            condition,
            body,
            [i, context_array]
        )

        context_vectors = context_array.stack()
        context_vectors = tf.transpose(
            context_vectors,
            [1, 0, 2]
        )

        # Combine decoder states with attention context
        combined = tf.concat(
            [decoder_outputs, context_vectors],
            axis=-1
        )

        predictions = self.decoder_dense(combined)

        return predictions

In [17]:
# Create padded datasets and the masked cross-entropy loss

BATCH_SIZE = 16

train_dataset = tf.data.Dataset.from_generator(
    lambda: zip(train_en, train_fr),
    output_signature=(
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    )
)

train_dataset = (
    train_dataset
    .shuffle(10000)
    .padded_batch(
        BATCH_SIZE,
        padded_shapes=([None], [None]),
        padding_values=(0, 0)
    )
    .prefetch(tf.data.AUTOTUNE)
)

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    reduction="none"
)

def loss_function(real, pred):
    loss = loss_object(real, pred)

    mask = tf.cast(real != 0, loss.dtype)
    loss *= mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

print("Dataset and loss function ready.")

Dataset and loss function ready.


In [18]:
# Create padded datasets and the masked cross-entropy loss

BATCH_SIZE = 16

train_dataset = tf.data.Dataset.from_generator(
    lambda: zip(train_en, train_fr),
    output_signature=(
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    )
)

train_dataset = (
    train_dataset
    .shuffle(10000)
    .padded_batch(
        BATCH_SIZE,
        padded_shapes=([None], [None]),
        padding_values=(0, 0)
    )
    .prefetch(tf.data.AUTOTUNE)
)

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=False,
    reduction="none"
)

def loss_function(real, pred):
    loss = loss_object(real, pred)

    mask = tf.cast(real != 0, loss.dtype)
    loss *= mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

print("Dataset and loss function ready.")

Dataset and loss function ready.


In [19]:
# Create the attention model and optimizer

attention_model = AttentionSeq2Seq(
    eng_vocab_size=len(eng_vocab),
    fra_vocab_size=len(fra_vocab),
    embedding_dim=EMBEDDING_DIM,
    units=UNITS
)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

print("Attention model created.")

Attention model created.


Will Now do One batch Testing

In [20]:
# Test the attention model on one training batch

english_batch, french_batch = next(iter(train_dataset))

decoder_input = french_batch[:, :-1]
target = french_batch[:, 1:]

predictions = attention_model(english_batch, decoder_input)

test_loss = loss_function(target, predictions)

print("English batch shape:", english_batch.shape)
print("French decoder input shape:", decoder_input.shape)
print("Predictions shape:", predictions.shape)
print("Test loss:", float(test_loss))

English batch shape: (16, 45)
French decoder input shape: (16, 49)
Predictions shape: (16, 49, 41836)
Test loss: 10.641583442687988


Now let's create the training step and benchmark 100 batches before deciding how many epochs we can afford.

In [21]:
# Create the training step for the attention model

@tf.function(reduce_retracing=True)
def attention_train_step(english_batch, french_batch):
    decoder_input = french_batch[:, :-1]
    target = french_batch[:, 1:]

    with tf.GradientTape() as tape:
        predictions = attention_model(english_batch, decoder_input)
        loss = loss_function(target, predictions)

    gradients = tape.gradient(
        loss,
        attention_model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, attention_model.trainable_variables)
    )

    return loss

In [22]:
# Benchmark attention training speed on 100 batches

import time

start_time = time.time()
total_loss = 0.0
batch_count = 0

for english_batch, french_batch in train_dataset.take(100):
    loss = attention_train_step(english_batch, french_batch)
    total_loss += float(loss)
    batch_count += 1

elapsed = time.time() - start_time

print(f"100 batches completed")
print(f"Average loss: {total_loss / batch_count:.4f}")
print(f"Time: {elapsed / 60:.2f} minutes")

100 batches completed
Average loss: 7.2185
Time: 0.40 minutes


Training on 1 epoch

In [23]:
# Train the attention model for one full epoch

import time

start_time = time.time()
total_loss = 0.0
batch_count = 0

for english_batch, french_batch in train_dataset:
    loss = attention_train_step(english_batch, french_batch)

    total_loss += float(loss)
    batch_count += 1

    if batch_count % 1000 == 0:
        print(
            f"Batch {batch_count} | "
            f"Average loss: {total_loss / batch_count:.4f}"
        )

epoch_loss = total_loss / batch_count
elapsed = time.time() - start_time

print(f"\nEpoch 1 loss: {epoch_loss:.4f}")
print(f"Epoch 1 time: {elapsed / 60:.2f} minutes")

Batch 1000 | Average loss: 5.7284
Batch 2000 | Average loss: 5.4028
Batch 3000 | Average loss: 5.1313
Batch 4000 | Average loss: 4.8862
Batch 5000 | Average loss: 4.6599
Batch 6000 | Average loss: 4.4541
Batch 7000 | Average loss: 4.2669
Batch 8000 | Average loss: 4.1035
Batch 9000 | Average loss: 3.9689
Batch 10000 | Average loss: 3.8589
Batch 11000 | Average loss: 3.7662
Batch 12000 | Average loss: 3.6871
Batch 13000 | Average loss: 3.6224
Batch 14000 | Average loss: 3.5614

Epoch 1 loss: 3.5279
Epoch 1 time: 41.37 minutes


Loss is decreasing hence ttention model is clearly learning rather than getting stuck.

In [26]:
# Check whether the Subtask 1 model objects still exist
print("encoder_model" in globals())
print("decoder_model" in globals())
print("model" in globals())

False
False
False


**Attention Greedy Decoder**

In [27]:
# Generate a French translation from the trained attention model
def translate_attention(sentence, max_length=50):
    english_ids = encode_sentence(sentence, eng_vocab)
    encoder_input = tf.constant([english_ids])

    # Start with <SOS>
    generated_ids = [fra_vocab["<SOS>"]]

    for _ in range(max_length):
        decoder_input = tf.constant([generated_ids])

        # Run the attention model on the generated sequence so far
        predictions = attention_model(encoder_input, decoder_input)

        # Get the prediction for the newest decoder position
        next_token = int(tf.argmax(predictions[0, -1]).numpy())

        if next_token == fra_vocab["<EOS>"]:
            break

        generated_ids.append(next_token)

    return generated_ids[1:]

In [28]:
# Test greedy decoding with the attention model
test_sentence = test_df.iloc[0]["en"]

generated_ids = translate_attention(test_sentence)

fra_id_to_token = {idx: token for token, idx in fra_vocab.items()}

generated_tokens = [
    fra_id_to_token.get(token_id, "<UNK>")
    for token_id in generated_ids
]

print("English:", test_sentence)
print("Generated French:", " ".join(generated_tokens))


English: Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
Generated French: il y a plusieurs ann es , voici la pr sentation ted , peter <UNK> a d cid de d fier le d fi du d fi .


Calculate Attention BLEU on first 100

In [29]:
# Calculate BLEU for the attention model on the first 100 test examples
from nltk.translate.bleu_score import corpus_bleu

references = []
hypotheses = []

for i in range(100):
    english_sentence = test_df.iloc[i]["en"]
    reference_sentence = test_df.iloc[i]["fr"]

    generated_ids = translate_attention(english_sentence)

    generated_tokens = [
        fra_id_to_token.get(token_id, "<UNK>")
        for token_id in generated_ids
    ]

    reference_tokens = tokenize(reference_sentence)

    references.append([reference_tokens])
    hypotheses.append(generated_tokens)

attention_bleu = corpus_bleu(references, hypotheses)

print(f"Attention model BLEU: {attention_bleu:.6f}")

Attention model BLEU: 0.212098


## Subtask 2 — Current Results

Subtask 2 extends the basic encoder-decoder from Subtask 1 by adding **Bahdanau attention**. The attention mechanism allows the decoder to access all encoder hidden states instead of relying only on a single fixed context vector.

### Training

The attention model was trained for one full epoch on the same English–French dataset used in Subtask 1.

| Metric | Result |
|---|---:|
| Epoch 1 Loss | **3.5279** |
| Training Time | **41.37 min** |

The training loss decreased steadily throughout the epoch, indicating that the model was learning.

### BLEU Comparison

Both models were evaluated on the **same first 100 test examples**.

| Model | BLEU |
|---|---:|
| Subtask 1 — No Attention | 0.067696 |
| Subtask 2 — Bahdanau Attention | **0.212098** |

The attention model achieved approximately **3.14× higher BLEU** than the no-attention baseline.

### Observation

Attention significantly improved translation quality. By attending to different encoder states during decoding, the model can better preserve information from different parts of the input sentence.

However, the generated translations can still contain:

- `<UNK>` tokens
- fragmented words
- repetitions
- incorrect or missing information

The next step is to investigate whether a different **decoding strategy**, specifically **beam search**, can further improve the generated translations.


## Implementing Bean Search

In [30]:
# Generate a translation using beam search with Bahdanau attention
def beam_search_attention(sentence, beam_width=3, max_length=50):
    english_ids = encode_sentence(sentence, eng_vocab)
    encoder_input = tf.constant([english_ids])

    sos_id = fra_vocab["<SOS>"]
    eos_id = fra_vocab["<EOS>"]

    # Each beam stores (token_ids, cumulative_log_probability)
    beams = [([sos_id], 0.0)]

    for _ in range(max_length):
        candidates = []

        for token_ids, score in beams:
            # Stop expanding sequences that already reached EOS
            if token_ids[-1] == eos_id:
                candidates.append((token_ids, score))
                continue

            decoder_input = tf.constant([token_ids])

            predictions = attention_model(
                encoder_input,
                decoder_input
            )

            # Get probabilities for the next token
            next_probs = predictions[0, -1]

            # Use log probabilities for numerical stability
            log_probs = tf.math.log(next_probs + 1e-10)

            # Get the best beam_width next tokens
            top_probs, top_ids = tf.math.top_k(
                log_probs,
                k=beam_width
            )

            for log_prob, token_id in zip(
                top_probs.numpy(),
                top_ids.numpy()
            ):
                new_tokens = token_ids + [int(token_id)]
                new_score = score + float(log_prob)

                candidates.append((new_tokens, new_score))

        # Keep the best beam_width candidates
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_width]

        # Stop if every beam has reached EOS
        if all(tokens[-1] == eos_id for tokens, _ in beams):
            break

    # Return the highest-scoring sequence
    best_tokens, best_score = beams[0]

    return best_tokens[1:]


In [31]:
# Test beam search on the same example used for greedy decoding
generated_ids = beam_search_attention(
    test_sentence,
    beam_width=3
)

generated_tokens = [
    fra_id_to_token.get(token_id, "<UNK>")
    for token_id in generated_ids
]

print("English:", test_sentence)
print("Beam search French:", " ".join(generated_tokens))

English: Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
Beam search French: il y a plusieurs ann es ted ici , peter <UNK> a pr sent une conception de design appel le d fi du d fi . <EOS>


We need to do some clean ups as output contains EOS and UNK

### Now lets calculate BLEU for beam search on the smae first 100 test examples

In [32]:
# Calculate BLEU for beam search on the same first 100 test examples
references = []
hypotheses = []

for i in range(100):
    english_sentence = test_df.iloc[i]["en"]
    reference_sentence = test_df.iloc[i]["fr"]

    generated_ids = beam_search_attention(
        english_sentence,
        beam_width=3
    )

    generated_tokens = [
        fra_id_to_token.get(token_id, "<UNK>")
        for token_id in generated_ids
        if token_id != fra_vocab["<EOS>"]
    ]

    reference_tokens = tokenize(reference_sentence)

    references.append([reference_tokens])
    hypotheses.append(generated_tokens)

beam_bleu = corpus_bleu(references, hypotheses)

print(f"Beam Search BLEU: {beam_bleu:.6f}")

Beam Search BLEU: 0.215984


Attention produced the major improvement, while beam search provided only a modest additional gain on this evaluation subset.

## generate the 5 qualitative examples.

In [33]:
# Compare greedy and beam-search translations on five test examples
for i in range(5):
    english_sentence = test_df.iloc[i]["en"]
    reference_sentence = test_df.iloc[i]["fr"]

    greedy_ids = translate_attention(
        english_sentence,
        max_length=50
    )

    beam_ids = beam_search_attention(
        english_sentence,
        beam_width=3,
        max_length=50
    )

    greedy_tokens = [
        fra_id_to_token.get(token_id, "<UNK>")
        for token_id in greedy_ids
        if token_id != fra_vocab["<EOS>"]
    ]

    beam_tokens = [
        fra_id_to_token.get(token_id, "<UNK>")
        for token_id in beam_ids
        if token_id != fra_vocab["<EOS>"]
    ]

    print(f"\n{'='*80}")
    print(f"Example {i+1}")
    print("English:   ", english_sentence)
    print("Reference: ", reference_sentence)
    print("Greedy:    ", " ".join(greedy_tokens))
    print("Beam:      ", " ".join(beam_tokens))


Example 1
English:    Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
Reference:  Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.
Greedy:     il y a plusieurs ann es , voici la pr sentation ted , peter <UNK> a d cid de d fier le d fi du d fi .
Beam:       il y a plusieurs ann es ted ici , peter <UNK> a pr sent une conception de design appel le d fi du d fi .

Example 2
English:    And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.
Reference:  Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow.
Greedy:     et les quipes <UNK> , les quipes de quatre ont de construire la stru

## Qualitative Analysis and Failure Modes

The generated translations show that Bahdanau attention improves the model's ability to preserve the overall structure and meaning of the input, but the model still makes substantial translation errors.

### Observations

- **Better contextual alignment:** Attention helps the decoder access information from different parts of the English sentence, producing more coherent sentence structures.
- **Rare-word and vocabulary errors:** The model frequently produces `<UNK>` tokens and fragmented words such as `ann es` and `pr sent`.
- **Semantic drift:** Some important words are replaced by unrelated words. For example, `marshmallow` is incorrectly generated as words such as `tigre`, `barbier`, or `cochon`.
- **Repetition:** The model sometimes repeats phrases or ideas, particularly for longer sentences.
- **Long-sentence degradation:** Sentences containing many details, objects, and quantities are still difficult for the model to translate accurately.
- **Beam search is only modestly better:** Beam search improved BLEU from 0.212098 to 0.215984, but the qualitative outputs show that it does not consistently produce better translations than greedy decoding.

### Overall Analysis

The main improvement in Subtask 2 comes from **attention rather than the decoding strategy**. Bahdanau attention increased BLEU substantially compared with the no-attention baseline, while beam search provided only a small additional improvement.

The remaining errors motivate further improvements such as better vocabulary handling, longer training, and more advanced architectures.

# Subtask 2 — Final Results and Takeaways

## Experimental Results

The basic encoder-decoder from Subtask 1 was extended with **Bahdanau additive attention**. The attention model was trained for one epoch on the same English-French dataset.

| Model | Decoding Strategy | BLEU |
|---|---|---:|
| No Attention | Greedy | 0.067696 |
| Bahdanau Attention | Greedy | **0.212098** |
| Bahdanau Attention | Beam Search (3) | **0.215984** |

The attention model achieved a substantial improvement over the no-attention baseline, increasing BLEU from **0.067696 to 0.212098**. This demonstrates the benefit of allowing the decoder to attend to different encoder states instead of relying on a single fixed context vector.

Beam search provided a smaller additional improvement, increasing BLEU from **0.212098 to 0.215984**. The qualitative examples also show that beam search does not consistently produce better translations than greedy decoding.

## Key Takeaways

1. **Attention was the major improvement.** It helped the decoder preserve information from different parts of the source sentence.
2. **Beam search provided only a modest improvement** on this evaluation set.
3. The model still struggled with **rare words, `<UNK>` tokens, fragmented words, repetition, semantic drift, and long information-dense sentences**.
4. These limitations motivate moving beyond single-sentence translation toward models that can use multiple sources of context.

## Conclusion

Subtask 2 demonstrated that adding Bahdanau attention substantially improves the basic sequence-to-sequence translation model. The comparison between greedy decoding and beam search showed that decoding strategy can provide additional gains, although the improvement was relatively small compared with the benefit obtained from attention.